# Broker Portfolio Agent — debug notebook

Interactive scratchpad for poking at the broker client, tools, and the LangGraph agent without going through the CLI or FastAPI server.

Run cells top to bottom. If you edit `app/*.py`, re-run the import cell (the autoreload extension below picks up changes automatically).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Ensure the project root (this notebook's directory) is importable as `app.*`
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.config import settings

print(f"LLM provider: {settings.llm_provider}")
print(f"Using mock broker data: {settings.use_mock_broker}")

LLM provider: groq
Using mock broker data: True


## 1. Poke at the broker client directly

Bypasses the LLM entirely — useful for checking `app/broker_client.py` output (mock or, once wired up, live) matches the Pydantic models in `app/models.py`.

In [2]:
from app.broker_client import broker_client

positions = broker_client.get_positions()
positions

[Position(symbol='AAPL', quantity=150.0, avg_cost=187.32, current_price=194.1),
 Position(symbol='NVDA', quantity=40.0, avg_cost=118.5, current_price=131.2),
 Position(symbol='MSFT', quantity=60.0, avg_cost=402.1, current_price=398.55)]

In [3]:
broker_client.get_account_summary()

AccountSummary(net_liquidation=182450.0, total_cash=41200.0, buying_power=82400.0, unrealized_pnl=3115.5, realized_pnl_today=-210.0)

In [4]:
broker_client.get_open_orders()

[OpenOrder(order_id=1001, symbol='TSLA', action='BUY', order_type='LMT', quantity=20.0, limit_price=245.0, status='Submitted')]

## 2. Call tools directly

Each `@tool`-decorated function in `app/tools.py` is invokable via `.invoke({})` without going through the agent — good for checking the formatted string output the LLM actually sees.

In [5]:
from app.tools import get_positions, get_account_summary, get_open_orders

print(get_positions.invoke({}))

AAPL: 150.0 shares, avg cost $187.32, current $194.10, unrealized P&L +3.6%, market value $29,115.00
NVDA: 40.0 shares, avg cost $118.50, current $131.20, unrealized P&L +10.7%, market value $5,248.00
MSFT: 60.0 shares, avg cost $402.10, current $398.55, unrealized P&L -0.9%, market value $23,913.00


## 3. Run the full LangGraph agent

This calls the configured LLM (Groq by default) and requires `GROQ_API_KEY` in `.env`.

In [6]:
from app.agent import ask

answer = ask("what are my current positions?", thread_id="debug-session")
print(answer)

Here are the positions currently held in your Interactive Brokers account:

| Symbol | Shares | Avg. Cost | Current Price | Unrealized P&L | Market Value |
|--------|--------|----------|---------------|----------------|--------------|
| **AAPL** | 150.0 | $187.32 | $194.10 | **+3.6 %** | **$29,115.00** |
| **NVDA** | 40.0 | $118.50 | $131.20 | **+10.7 %** | **$5,248.00** |
| **MSFT** | 60.0 | $402.10 | $398.55 | **‑0.9 %** | **$23,913.00** |

**Total market value of positions:** **$58,276.00**

These figures reflect the latest market data available to the read‑only view of your account. Let me know if you’d like any additional details (e.g., account summary, recent fills, open orders, or bracket‑order status).


In [7]:
# Follow-up in the same thread — checks memory/statefulness across turns
answer = ask("which of those are up more than 5%?", thread_id="debug-session")
print(answer)

Only one of your current positions is up more than 5%:

| Symbol | Shares | Avg. Cost | Current Price | Unrealized P&L |
|--------|--------|----------|---------------|----------------|
| **NVDA** | 40.0 | $118.50 | $131.20 | **+10.7 %** |

AAPL (+3.6 %) and MSFT (‑0.9 %) are both below the 5 % threshold. Let me know if you’d like any further details.


## 4. Inspect the raw graph state

Drop below `ask()` to see the full message list (including tool calls and tool outputs) for a given thread — handy when the final answer looks wrong and you need to see what the model actually called.

In [8]:
from app.agent import agent

config = {"configurable": {"thread_id": "debug-session"}}
state = agent.get_state(config)
for m in state.values["messages"]:
    print(f"--- {m.type} ---")
    print(m.content)
    if getattr(m, "tool_calls", None):
        print("tool_calls:", m.tool_calls)
    print()

--- human ---
what are my current positions?

--- ai ---

tool_calls: [{'name': 'get_positions', 'args': {}, 'id': 'fc_2f7a2549-c4fd-48c8-8e20-7180de37e89f', 'type': 'tool_call'}]

--- tool ---
AAPL: 150.0 shares, avg cost $187.32, current $194.10, unrealized P&L +3.6%, market value $29,115.00
NVDA: 40.0 shares, avg cost $118.50, current $131.20, unrealized P&L +10.7%, market value $5,248.00
MSFT: 60.0 shares, avg cost $402.10, current $398.55, unrealized P&L -0.9%, market value $23,913.00

--- ai ---
Here are the positions currently held in your Interactive Brokers account:

| Symbol | Shares | Avg. Cost | Current Price | Unrealized P&L | Market Value |
|--------|--------|----------|---------------|----------------|--------------|
| **AAPL** | 150.0 | $187.32 | $194.10 | **+3.6 %** | **$29,115.00** |
| **NVDA** | 40.0 | $118.50 | $131.20 | **+10.7 %** | **$5,248.00** |
| **MSFT** | 60.0 | $402.10 | $398.55 | **‑0.9 %** | **$23,913.00** |

**Total market value of positions:** **$58,276